# Stage 6 – Fairness on accepted loans

Group-level predicted default rates and disparate impact ratios by purpose, home ownership, employment length and grade.

**v2 fix:** drops post-outcome columns (`last_fico_range_*`, `debt_settlement_flag`, settlement fields) and `id`, and splits train/test before target encoding, matching notebook 03.


In [1]:
# 05_fairness_accepted.ipynb
# ============================================
# Fairness Analysis on Accepted Loans Dataset
# Project: Explainable AI for Credit Risk
# ============================================

import os
import gc
import math
import random
import warnings

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    auc,
)

import xgboost as xgb
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

# -------------------------------------------------
# 0. Config
# -------------------------------------------------
DATA_CANDIDATES = [
    "../data/accepted_cleaned.csv",                                   # repo layout
    r"D:\credit-risk-xai\credit-risk-xai\data\accepted_cleaned.csv",  # my local copy
]
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), DATA_CANDIDATES[0])
SAVE_DIR = "../outputs/notebook5_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_FOLDS_FOR_TE = 5
SAMPLE_TARGET = 200_000       # sample size for fairness analysis (optional)

TARGET_COL = "default"        # created in Notebook 1 (1 = charged off, 0 = fully paid)

print("Using data:", DATA_PATH)
print("Outputs will be stored in:", SAVE_DIR)

# -------------------------------------------------
# 1. Memory-safe sampling of accepted_cleaned.csv
# -------------------------------------------------
def reservoir_sample_csv(path: str, sample_size: int, random_state: int = 42, chunksize: int = 100_000):
    """
    Memory-safe reservoir sample from large CSV.
    Returns: sampled DataFrame (size ~sample_size), total rows read.
    """
    rng = random.Random(random_state)
    reservoir = []
    total = 0

    reader = pd.read_csv(path, chunksize=chunksize, low_memory=True)
    for chunk in tqdm(reader, desc="Streaming accepted_cleaned.csv"):
        for _, row in chunk.iterrows():
            if total < sample_size:
                reservoir.append(row)
            else:
                j = rng.randrange(total + 1)
                if j < sample_size:
                    reservoir[j] = row
            total += 1
        del chunk
        gc.collect()

    if total == 0:
        raise FileNotFoundError(f"No rows read from {path}.")

    sampled_df = pd.DataFrame(reservoir)
    sampled_df.reset_index(drop=True, inplace=True)
    return sampled_df, total

print("\n1) Sampling accepted_cleaned.csv in a memory-safe way...")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}")

accepted_sample, total_rows = reservoir_sample_csv(DATA_PATH, SAMPLE_TARGET, random_state=RANDOM_STATE)
print(f"Sampled {len(accepted_sample)} rows from {total_rows} total rows.")

df = accepted_sample.copy()
print("Sample columns:", df.columns[:15])

# -------------------------------------------------
# 2. Target & leakage handling
# -------------------------------------------------
print("\n2) Preparing target and dropping payment/leakage columns...")

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in accepted_cleaned.csv")

df[TARGET_COL] = df[TARGET_COL].astype(int)

# Drop explicit strong leakage: post-outcome payment information etc.
leakage_cols = [
    "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "last_pymnt_amnt", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "next_pymnt_d", "last_credit_pull_d", "loan_status",
    # v2: also drop columns only known after the loan outcome, and identifiers
    "out_prncp", "out_prncp_inv", "debt_settlement_flag",
    "settlement_status", "settlement_date", "settlement_amount", "settlement_percentage",
    "last_fico_range_low", "last_fico_range_high",
    "id", "desc", "url", "title", "zip_code", "addr_state", "policy_code",
]

cols_to_drop = [c for c in leakage_cols if c in df.columns]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print("Dropped leakage columns:", cols_to_drop)

# Drop obviously constant / all-null columns
nunique = df.nunique(dropna=True)
const_cols = [c for c in df.columns if c != TARGET_COL and nunique.get(c, 0) <= 1]
if const_cols:
    df.drop(columns=const_cols, inplace=True)
    print("Dropped constant/all-null columns:", const_cols)

print("Remaining columns:", len(df.columns))

# -------------------------------------------------
# 3. Detect numeric vs categorical features
# -------------------------------------------------
print("\n3) Detecting numeric vs categorical features...")

all_features = [c for c in df.columns if c != TARGET_COL]

numeric_cols = []
categorical_cols = []

for c in all_features:
    # quick sample to infer type
    sample_vals = df[c].dropna().head(500)
    # try numeric
    numeric_conv = pd.to_numeric(sample_vals, errors="coerce")
    non_numeric_fraction = numeric_conv.isna().sum() / max(1, len(numeric_conv))

    # heuristic: if dtype object or mostly non-numeric or low cardinality -> treat as categorical
    if df[c].dtype == object or non_numeric_fraction > 0.5 or df[c].nunique() < 40:
        categorical_cols.append(c)
    else:
        numeric_cols.append(c)

print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print("Example numeric:", numeric_cols[:10])
print("Example categorical:", categorical_cols[:10])

# Coerce numeric columns to float
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Fill NaNs in categoricals with a missing token
df[categorical_cols] = df[categorical_cols].fillna("__MISSING__").astype(str)

# -------------------------------------------------
# 4. Split categorical into small vs large cardinality
# -------------------------------------------------
print("\n4) Splitting categorical columns by cardinality...")
small_cat = [c for c in categorical_cols if df[c].nunique() < 20]
large_cat = [c for c in categorical_cols if df[c].nunique() >= 20]

print("Small-cardinality categoricals:", small_cat)
print("Large-cardinality categoricals (for target encoding):", large_cat)

# Optional cap on number of large-cardinality columns to keep TE quick
MAX_LARGE_CAT = 5
if len(large_cat) > MAX_LARGE_CAT:
    # keep the most frequent ones
    freq_sorted = sorted(large_cat, key=lambda c: df[c].value_counts().head(100).sum(), reverse=True)
    keep_large = freq_sorted[:MAX_LARGE_CAT]
    drop_large = [c for c in large_cat if c not in keep_large]
    print(f"Reducing TE columns to top {MAX_LARGE_CAT}: keeping {keep_large}, dropping {drop_large}")
    large_cat = keep_large

# -------------------------------------------------
# 5. Train/test split FIRST, then target encoding fitted on train only
#    (v2 fix: previously test rows were encoded using labels from the whole sample)
# -------------------------------------------------
print("\n5) Train/test split, then leak-free target encoding...")

y_series = df[TARGET_COL].astype(int)
train_idx, test_idx = train_test_split(
    df.index.to_numpy(),
    test_size=TEST_SIZE,
    stratify=y_series,
    random_state=RANDOM_STATE,
)
train_mask = df.index.isin(train_idx)
global_mean = y_series.loc[train_idx].mean()
TE_SMOOTHING = 20

def smoothed_means(frame, col):
    stats = frame.groupby(col)[TARGET_COL].agg(["mean", "count"])
    return (stats["mean"] * stats["count"] + global_mean * TE_SMOOTHING) / (stats["count"] + TE_SMOOTHING)

X_te = pd.DataFrame(index=df.index)
train_df = df.loc[train_idx]
kf = StratifiedKFold(n_splits=N_FOLDS_FOR_TE, shuffle=True, random_state=RANDOM_STATE)
for col in large_cat:
    print(f"  TE -> {col}")
    enc = pd.Series(global_mean, index=df.index, dtype=float)
    # train rows: out-of-fold within the training set
    for fit_pos, enc_pos in kf.split(train_df, y_series.loc[train_idx]):
        means = smoothed_means(train_df.iloc[fit_pos], col)
        rows = train_df.index[enc_pos]
        enc.loc[rows] = df.loc[rows, col].map(means).fillna(global_mean).values
    # test rows: means from the full training set only
    full_means = smoothed_means(train_df, col)
    enc.loc[test_idx] = df.loc[test_idx, col].map(full_means).fillna(global_mean).values
    X_te[col + "_te"] = enc

if large_cat:
    df.drop(columns=large_cat, inplace=True)

# -------------------------------------------------
# 6. Ordinal encoding for small-cardinality categoricals (fit on train only)
# -------------------------------------------------
print("\n6) Ordinal encoding small-cardinality categoricals...")
if small_cat:
    ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    ordinal_encoder.fit(df.loc[train_idx, small_cat].astype(str))
    df_small_ord = pd.DataFrame(
        ordinal_encoder.transform(df[small_cat].astype(str)),
        columns=[c + "_ord" for c in small_cat],
        index=df.index,
    )
else:
    ordinal_encoder = None
    df_small_ord = pd.DataFrame(index=df.index)

# -------------------------------------------------
# 7. Assemble feature matrix (index kept so fairness groups line up)
# -------------------------------------------------
print("\n7) Assembling final feature matrix...")
X_all = pd.concat([df[numeric_cols], df_small_ord, X_te], axis=1)
y_all = y_series
feature_names = list(X_all.columns)
print("Final feature matrix:", X_all.shape)

# -------------------------------------------------
# 8. Apply the split & numeric preprocessing
# -------------------------------------------------
print("\n8) Applying split and numeric preprocessing...")
X_train_df, X_test_df = X_all.loc[train_idx], X_all.loc[test_idx]
y_train, y_test = y_all.loc[train_idx], y_all.loc[test_idx]
print("Train:", X_train_df.shape, "Test:", X_test_df.shape)

numeric_cols_final = numeric_cols.copy()

preprocessor_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

if len(numeric_cols_final) > 0:
    X_train_num = preprocessor_num.fit_transform(X_train_df[numeric_cols_final])
    X_test_num = preprocessor_num.transform(X_test_df[numeric_cols_final])
else:
    X_train_num = np.zeros((X_train_df.shape[0], 0))
    X_test_num = np.zeros((X_test_df.shape[0], 0))

other_cols = [c for c in X_all.columns if c not in numeric_cols_final]

X_train_other = X_train_df[other_cols].to_numpy(dtype=float) if other_cols else np.zeros((X_train_df.shape[0], 0))
X_test_other = X_test_df[other_cols].to_numpy(dtype=float) if other_cols else np.zeros((X_test_df.shape[0], 0))

X_train_final = np.hstack([X_train_num, X_train_other])
X_test_final = np.hstack([X_test_num, X_test_other])

final_feature_names = (numeric_cols_final + other_cols)
print("Final train matrix:", X_train_final.shape)
print("Final test matrix:", X_test_final.shape)

# -------------------------------------------------
# 9. Train XGBoost model (or load from Notebook 3)
# -------------------------------------------------
print("\n9) Training XGBoost model for fairness analysis...")

# 👉 If you want to reuse the xgb_clf from Notebook 3 instead of retraining,


xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_estimators=250,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8
)

xgb_clf.fit(X_train_final, y_train)

# Evaluation
probs_test = xgb_clf.predict_proba(X_test_final)[:, 1]
y_pred = (probs_test >= 0.5).astype(int)

print("\n=== XGBoost performance on held-out test ===")
print(classification_report(y_test, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
roc = roc_auc_score(y_test, probs_test)
prec, rec, _ = precision_recall_curve(y_test, probs_test)
pr_auc = auc(rec, prec)
print(f"ROC AUC: {roc:.6f}, PR AUC: {pr_auc:.6f}")

# -------------------------------------------------
# 10. Fairness Analysis – reattach group attributes
# -------------------------------------------------
print("\n10) Fairness analysis on TEST SET by group attributes...")

# We need group columns (purpose, home_ownership, grade, emp_length).
# They are still present in the original sampled df before we dropped / transformed.

# Re-load the same SAMPLE rows (simplest = reuse 'df_raw' before dropping anything).
# Here we still have 'accepted_sample' in memory with all original columns.
df_raw = accepted_sample.reset_index(drop=True)   # original columns, including groups

# Align indices: X_all and df_raw have same index order before train_test_split.
# After split, X_test_df has subset of indices; we use those to slice df_raw.
test_idx = X_test_df.index
df_test_groups = df_raw.loc[test_idx, :].copy()
df_test_groups = df_test_groups.reset_index(drop=True)

# Also align predictions and true labels to this reset index
df_fair = df_test_groups.copy()
df_fair["true_default"] = y_test.reset_index(drop=True)
df_fair["pred_default_prob"] = probs_test
df_fair["pred_label"] = y_pred

print("Fairness dataframe shape:", df_fair.shape)
print("Columns for groups example:", df_fair.columns[:20])

# -------------------------------------------------
# 11. Helper: fairness stats by group
# -------------------------------------------------
def fairness_stats(df, group_col):
    """
    Compute fairness stats per group:
    - count
    - average predicted default probability
    - default rate (true)
    - predicted "high-risk" rate (pred_label=1)
    """
    if group_col not in df.columns:
        print(f"⚠️ Column {group_col} not found. Skipping.")
        return None

    groups = df[group_col].fillna("__MISSING__").astype(str).unique()
    rows = []

    for g in groups:
        mask = (df[group_col].astype(str) == str(g))
        n = mask.sum()
        if n < 200:   # skip tiny groups to avoid noise
            continue

        avg_prob = df.loc[mask, "pred_default_prob"].mean()
        true_rate = df.loc[mask, "true_default"].mean()
        reject_rate = df.loc[mask, "pred_label"].mean()

        rows.append({
            "group": g,
            "count": int(n),
            "avg_pred_prob": avg_prob,
            "true_default_rate": true_rate,
            "predicted_default_rate": reject_rate
        })

    if not rows:
        print(f"No valid groups (>=200 samples) for {group_col}.")
        return None

    stats_df = pd.DataFrame(rows).sort_values("avg_pred_prob", ascending=False)
    return stats_df

# -------------------------------------------------
# 12. Run fairness slices
# -------------------------------------------------
group_cols = ["purpose", "home_ownership", "emp_length", "grade"]

for col in group_cols:
    print(f"\n=== Fairness analysis by: {col} ===")
    stats = fairness_stats(df_fair, col)
    if stats is not None:
        print(stats)
        out_path = os.path.join(SAVE_DIR, f"fairness_accepted_{col}.csv")
        stats.to_csv(out_path, index=False)
        print(f"Saved fairness stats to {out_path}")

# -------------------------------------------------
# 13. Simple disparity ratios (statistical parity)
# -------------------------------------------------
def disparate_impact(df, group_col):
    """
    Statistical parity / disparate impact:
    ratio = min(reject_rate) / max(reject_rate)
    < 0.8 often flagged as potential fairness issue.
    """
    if group_col not in df.columns:
        return None

    groups = df[group_col].fillna("__MISSING__").astype(str).unique()
    rates = {}

    for g in groups:
        mask = (df[group_col].astype(str) == str(g))
        if mask.sum() < 200:
            continue
        rates[g] = df.loc[mask, "pred_label"].mean()

    if len(rates) < 2:
        return None

    min_rate = min(rates.values())
    max_rate = max(rates.values())
    ratio = min_rate / max_rate if max_rate > 0 else 1.0
    return ratio, rates

for col in group_cols:
    print(f"\n=== Disparate Impact (Statistical Parity) for {col} ===")
    di = disparate_impact(df_fair, col)
    if di is None:
        print("Not enough group data for DI.")
    else:
        ratio, rates = di
        print("Group predicted default (rejection) rates:")
        for g, r in rates.items():
            print(f"  {g}: {r:.3f}")
        print(f"Disparate impact ratio (min/max): {ratio:.3f}")
        if ratio < 0.8:
            print("⚠️ Potential fairness issue (below 0.8 rule-of-thumb).")

# -------------------------------------------------
# 14. Optional: plot fairness results
# -------------------------------------------------
import matplotlib.pyplot as plt

def plot_fairness_bar(stats_df, group_col, metric_col, title_suffix=""):
    plt.figure(figsize=(10, 5))
    plt.bar(stats_df["group"].astype(str), stats_df[metric_col])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(metric_col)
    plt.title(f"{metric_col} by {group_col} {title_suffix}")
    plt.tight_layout()
    out_path = os.path.join(SAVE_DIR, f"{group_col}_{metric_col}.png")
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"Saved plot: {out_path}")

for col in group_cols:
    stats_path = os.path.join(SAVE_DIR, f"fairness_accepted_{col}.csv")
    if os.path.exists(stats_path):
        stats_df = pd.read_csv(stats_path)
        if "avg_pred_prob" in stats_df.columns:
            plot_fairness_bar(stats_df, col, "avg_pred_prob", "(accepted loans test set)")
        if "true_default_rate" in stats_df.columns:
            plot_fairness_bar(stats_df, col, "true_default_rate", "(ground truth)")
        if "predicted_default_rate" in stats_df.columns:
            plot_fairness_bar(stats_df, col, "predicted_default_rate", "(model decisions)")

print("\n✅ Notebook 5 (Fairness on accepted loans) completed.")
print("Check CSVs and plots in:", SAVE_DIR)


d:\anaconda\envs\credit-risk-xai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using data: D:\credit-risk-xai\credit-risk-xai\data\accepted_cleaned.csv
Outputs will be stored in: ../outputs/notebook5_outputs

1) Sampling accepted_cleaned.csv in a memory-safe way...


Streaming accepted_cleaned.csv: 14it [01:20,  5.78s/it]


Sampled 200000 rows from 1345310 total rows.
Sample columns: Index(['id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate',
       'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'purpose'],
      dtype='object')

2) Preparing target and dropping payment/leakage columns...
Dropped leakage columns: ['out_prncp', 'out_prncp_inv', 'debt_settlement_flag', 'last_fico_range_low', 'last_fico_range_high', 'id']
Dropped constant/all-null columns: ['hardship_flag']
Remaining columns: 86

3) Detecting numeric vs categorical features...
Numeric features: 48
Categorical features: 37
Example numeric: ['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate', 'installment', 'annual_inc', 'dti', 'mths_since_last_delinq', 'open_acc', 'revol_bal']
Example categorical: ['term', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'delinq_2yrs', 'earliest_cr_line'